# План проекта
В проекте вы реализуете pretrain и posttrain этапы обучения LLM. Выполняйте проект в Jupyter Notebook на ВМ. Выполнение заданий проекта займёт от 5 до 8 часов, не считая времени на обучение. 


## Pretrain
Претрейн — самый ресурсоёмкий этап обучения LLM. Чтобы полноценно обучить даже небольшую модель (менее 1B), понадобится более 10к GPU-часов на A100. Чтобы не тратить недели на обучение, но отработать ключевые приёмы, в проекте вы выполните упрощённую задачу. 

При полноценном претрейне модель учится обобщать знания из данных, на которых происходило обучение, чтобы потом извлекать эти знания по текстовым запросам уже после обучения. Упростим задачу — научим модель только структуре языка. 
Сосредоточимся на одном узком домене — текстах произведений русской литературы — и обучим модель продолжать фразы из этого домена разумным текстом. 

### Шаги этапа
1. Скачайте данные из [репозитория](https://github.com/JoannaBy/RussianNovels/tree/master/corpus) и упакуйте их в один датасет. Вам понадобятся все произведения из репозитория.

In [4]:
import os
from github_downloader import download_from_github
url = "https://github.com/JoannaBy/RussianNovels/tree/master/corpus"
current_dir = os.getcwd()
dataset_folder = os.path.join(current_dir, "dataset")
try:
    os.makedirs(dataset_folder, exist_ok=True)
    download_from_github(
        url=url,
        dest_folder=dataset_folder,
    )
    print(f"Датасет успешно загружен в: {dataset_folder}")    
except Exception as e:
    print(f"Произошла ошибка при загрузке: {e}")

Downloaded: /Users/papa/Documents/Практикум/Deep learning Engineer/dle_practicum/Sprint 6/Project/dataset/Bulgakov_BelayaGvardiya.txt
Downloaded: /Users/papa/Documents/Практикум/Deep learning Engineer/dle_practicum/Sprint 6/Project/dataset/Bulgakov_Diavoliada.txt
Downloaded: /Users/papa/Documents/Практикум/Deep learning Engineer/dle_practicum/Sprint 6/Project/dataset/Bulgakov_Master.txt
Downloaded: /Users/papa/Documents/Практикум/Deep learning Engineer/dle_practicum/Sprint 6/Project/dataset/Bulgakov_RokovyeYaytsa.txt
Downloaded: /Users/papa/Documents/Практикум/Deep learning Engineer/dle_practicum/Sprint 6/Project/dataset/Bulgakov_TeatralnyjRoman.txt
Downloaded: /Users/papa/Documents/Практикум/Deep learning Engineer/dle_practicum/Sprint 6/Project/dataset/Bulgakov_ZapiskiYonogoVracha.txt
Downloaded: /Users/papa/Documents/Практикум/Deep learning Engineer/dle_practicum/Sprint 6/Project/dataset/Chekhov_Dama.txt
Downloaded: /Users/papa/Documents/Практикум/Deep learning Engineer/dle_practicum

2. Проведите препроцессинг данных:
   - Очистите их от дубликатов.
   - Очистите от предложений с буквами не из кириллицы.
   - Обработайте повторяющуюся пунктуацию и т. д.
   - Разбейте на чанки поменьше, чтобы можно было добавить `<bos>` и `<eos>` токены в соответствии с обучаемой длиной контекста.

#### Препроцессинг

In [21]:
import os
import re
import glob
import json
from typing import List, Tuple, Set
from collections import defaultdict
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, processors
import numpy as np
from tqdm.auto import tqdm


def load_text_files(data_dir: str) -> List[str]:
    """
    Загружает все текстовые файлы из указанной директории.
    Возвращает список строк (каждая строка - содержимое файла).
    """
    texts = []
    file_pattern = os.path.join(data_dir, "*.txt")
    for file_path in tqdm(glob.glob(file_pattern)):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                text = f.read()
                texts.append(text)
        except Exception as e:
            print(f"Ошибка при чтении файла {file_path}: {e}")
    return texts


def split_into_sentences(text: str) -> List[str]:
    """
    Разбивает текст на предложения по точкам, восклицательным и вопросительным знакам.
    Учитывает многоточие и сокращения (например, "т.д.").
    Упрощённый подход.
    """
    # Заменяем многоточие на специальный маркер, чтобы не разбивать по нему
    text = re.sub(r'\.\.\.', '…', text)
    # Заменяем сокращения с точками
    abbreviations = [r'т\.д\.', r'т\.п\.', r'др\.', r'пр\.', r'г\.', r'см\.', r'ст\.', r'кн\.', r'ч\.', r'с\.']
    for abbr in abbreviations:
        text = re.sub(abbr, abbr.replace('.', '@'), text)
    
    # Разбиваем по . ! ? … (многоточие)
    sentences = re.split(r'(?<=[.!?…]) +', text)
    
    # Восстанавливаем сокращения
    restored = []
    for sent in tqdm(sentences):
        sent = sent.replace('@', '.')
        restored.append(sent.strip())
    
    # Убираем пустые предложения
    restored = [s for s in restored if s]
    return restored


def filter_cyrillic_sentences(sentences: List[str]) -> List[str]:
    """
    Оставляет только предложения, состоящие преимущественно из кириллических символов,
    пробелов, знаков пунктуации и цифр.
    """
    # Регулярное выражение для кириллических символов, пробелов, пунктуации и цифр
    cyrillic_pattern = re.compile(r'^[а-яёА-ЯЁ0-9\s\.,!?;:"\'\-–—()…]+$')
    filtered = []
    for sent in sentences:
        if cyrillic_pattern.match(sent):
            filtered.append(sent)
        else:
            # Можно также проверить процент кириллических символов
            # но для простоты используем строгое соответствие
            pass
    return filtered


def deduplicate_sentences(sentences: List[str]) -> List[str]:
    """
    Удаляет дубликаты предложений (точное совпадение).
    """
    seen = set()
    unique = []
    for sent in sentences:
        if sent not in seen:
            seen.add(sent)
            unique.append(sent)
    return unique


def clean_punctuation(text: str) -> str:
    """
    Очищает повторяющуюся пунктуацию (например, "!!!", "??", ",,").
    Заменяет множественные пробелы на один, удаляет неразрывные пробелы.
    """
    # Убираем повторяющиеся знаки препинания (оставляем один)
    text = re.sub(r'([!?])\1+', r'\1', text)  # !! -> !
    text = re.sub(r'(,)\1+', r'\1', text)     # ,, -> ,
    text = re.sub(r'(\.)\1+', r'\1', text)    # .. -> . (хотя .. обычно не встречается)
    # Убираем повторяющиеся дефисы, тире
    text = re.sub(r'(-)\1+', r'\1', text)
    text = re.sub(r'(—)\1+', r'\1', text)
    # Заменяем все whitespace символы (включая неразрывные пробелы, табуляции) на обычный пробел
    text = re.sub(r'\s+', ' ', text)
    # Удаляем пробелы перед знаками препинания (кроме открывающих скобок)
    text = re.sub(r'\s+([.,!?;:])', r'\1', text)
    # Удаляем пробелы после открывающих скобок и перед закрывающими
    text = re.sub(r'\(\s+', '(', text)
    text = re.sub(r'\s+\)', ')', text)
    return text.strip()


def chunk_text(sentences: List[str], max_chunk_size: int = 512) -> List[str]:
    """
    Объединяет предложения в чанки примерно max_chunk_size символов.
    Добавляет специальные токены <bos> и <eos> в начале и конце каждого чанка.
    """
    chunks = []
    current_chunk = []
    current_length = 0
    
    for sent in tqdm(sentences, "Формируем чанки: "):
        sent_len = len(sent)
        if current_length + sent_len + 1 <= max_chunk_size:  # +1 для пробела
            current_chunk.append(sent)
            current_length += sent_len + 1
        else:
            if current_chunk:
                chunk_text = ' '.join(current_chunk)
                chunk_text = f"<bos> {chunk_text} <eos>"
                chunks.append(chunk_text)
            # Начинаем новый чанк с текущим предложением
            current_chunk = [sent]
            current_length = sent_len
    
    # Добавляем последний чанк
    if current_chunk:
        chunk_text = ' '.join(current_chunk)
        chunk_text = f"<bos> {chunk_text} <eos>"
        chunks.append(chunk_text)
    
    return chunks

загрузим данные посмотрим на текст из списка текстов

In [15]:
import os
current_dir = os.getcwd()
dataset_folder = os.path.join(current_dir, "dataset")
text_files = load_text_files(dataset_folder)
print(text_files[0][:1000])

100%|██████████| 108/108 [00:00<00:00, 345.30it/s]

                           Не сохами-то славная землюшка наша распахана...
                           Распахана наша землюшка лошадиными копытами,
                           А засеяна славная землюшка казацкими головами,
                           Украшен-то наш тихий Дон молодыми вдовами,
                           Цветет наш батюшка тихий Дон сиротами,
                           Наполнена волна в тихом Дону отцовскими,
                           материнскими слезами.

                           Ой ты, наш батюшка тихий Дон!
                           Ой, что же ты, тихий Дон, мутнехонек течешь?
                           Ах, как мне, тихому Дону, не мутну течи!
                           Со дна меня, тиха Дона, студены ключи бьют,
                           Посередь меня, тиха Дона, бела рыбица мутит,

                                                   Старинные казачьи песни



 * КНИГА ПЕРВАЯ * 



 * ЧАСТЬ ПЕРВАЯ * 



I


   Мелеховский двор - на самом краю хутора.  Воротца  со  

сконкатенируем текста и разобъем их на предложения

In [18]:
sentences = split_into_sentences("\n".join(text_files))
print(f"Предложений: {len(sentences)}")
print()
for i in range(10):
    print(f"Предложение {i}")
    print(sentences[i])

100%|██████████| 379440/379440 [00:00<00:00, 2491166.51it/s]

Предложений: 379440

Предложение 0
Не сохами-то славная землюшка наша распахана…
                           Распахана наша землюшка лошадиными копытами,
                           А засеяна славная землюшка казацкими головами,
                           Украшен-то наш тихий Дон молодыми вдовами,
                           Цветет наш батюшка тихий Дон сиротами,
                           Наполнена волна в тихом Дону отцовскими,
                           материнскими слезами.

                           Ой ты, наш батюшка тихий Дон!
                           Ой, что же ты, тихий Дон, мутнехонек течешь?
                           Ах, как мне, тихому Дону, не мутну течи!
                           Со дна меня, тиха Дона, студены ключи бьют,
                           Посередь меня, тиха Дона, бела рыбица мутит,

                                                   Старинные казачьи песни



 * КНИГА ПЕРВАЯ * 



 * ЧАСТЬ ПЕРВАЯ * 



I


   Мелеховский двор - на самом краю хутора.
Предложе

In [22]:
processed = filter_cyrillic_sentences(sentences)
print(f"Предложений после фильтрации кириллицы: {len(processed)}")

processed = [clean_punctuation(sentence) for sentence in processed]
processed = deduplicate_sentences(processed)
print(f"Предложений после дедупликации: {len(processed)}")

chunks = chunk_text(processed, max_chunk_size=512)
print(f"Чанков: {len(chunks)}")   

Предложений после фильтрации кириллицы: 340805
Предложений после дедупликации: 326808


Формируем чанки: 100%|██████████| 326808/326808 [00:00<00:00, 2145859.45it/s]

Чанков: 82064


3. Создайте и обучите собственный токенизатор на полученных данных. Размер словаря выберите небольшим: при обучении только на рассмотренных текстах — около 3к токенов. В рассматриваемых данных язык намного менее разнообразен, чем в совокупных данных, поэтому крупные токенизаторы от реальных LLM могут не подойти. 

   При создании токенизатора можете ориентироваться на [материал huggingface.co](https://huggingface.co/learn/llm-course/ru/chapter6/8). Рекомендуем использовать BPE.

In [58]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from transformers import PreTrainedTokenizerFast


def train_bpe_tokenizer(texts: List[str], vocab_size: int = 3000, save_path: str = "tokenizer.json"):
    """
    Обучает BPE токенизатор на предоставленных текстах.
    Сохраняет токенизатор в файл.
    """    
    # Инициализируем токенизатор с BPE моделью
    tokenizer = Tokenizer(BPE(unk_token="<unk>"))
    tokenizer.pre_tokenizer = Whitespace()
    
    # Создаём тренера с указанным размером словаря
    trainer = BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=["<pad>", "<unk>", "<bos>", "<eos>", "<mask>"],
        min_frequency=2
    )
    
    # Обучаем на текстах (передаём список строк)
    tokenizer.train_from_iterator(texts, trainer)
    
    # Сохраняем токенизатор
    tokenizer.save(save_path)
    print(f"Токенизатор сохранён в {save_path}")

    # Возвращаем токенизатор для дальнейшего использования
    return tokenizer

In [64]:
tokenizer = train_bpe_tokenizer(sentences, vocab_size=3000, save_path="tokenizer.json")
tokenizer_fast = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer,
    unk_token="<unk>",
    pad_token="<pad>",
    bos_token="<bos>",
    eos_token="<eos>",
    mask_token="<mask>",
)

print(f"Размер словаря: {len(tokenizer.get_vocab())}")




Токенизатор сохранён в tokenizer.json
Размер словаря: 3000


In [65]:
def infer_tokenizer(tokenizer, texts):
    """Тестирует токенизатор на наборе предложений."""
    for i, text in enumerate(texts):
        encoded = tokenizer.encode(text)
        print(f"\nПредложение {i+1}: {text[:80]}...")
        print(f"   Токены: {encoded.tokens[:20]}{'...' if len(encoded.tokens) > 20 else ''}")
        print(f"   IDs:    {encoded.ids[:10]}{'...' if len(encoded.ids) > 10 else ''}")
        print(f"   Количество токенов: {len(encoded.tokens)}")


def specail_tokens_check(tokenizer): 
    """Проверка специальных токенов."""
    special_tokens = ["<bos>", "<eos>", "<pad>", "<unk>", "<mask>"]
    for tok in special_tokens:
        try:
            id_ = tokenizer.token_to_id(tok)
            print(f"  {tok}: ID = {id_}")
        except:
            print(f"  {tok}: не найден в словаре")


def count_unk(tokenizer, texts):
    """Тестирует встречаемость токена <unk>"""
    token_cnt, unk_cnt = 0, 0
    for text in tqdm(texts, "Скан текстов"):
        tokens = tokenizer.encode(text).tokens
        token_cnt += len(tokens)
        unk_cnt += len([t for t in tokens if t == '<unk>'])
    print(f"Кол-во <unk> токенов в текстах: {unk_cnt}")
    print(f"Доля <unk> токенов в текстах:   {100 * unk_cnt/token_cnt:.4f}")


print("\nПример работы токенизатора")
infer_tokenizer(tokenizer, chunks[10000:10003])

print("\nПроверка специальных токенов")
specail_tokens_check(tokenizer)

print("\nПроверка unk токенов")
count_unk(tokenizer, chunks)


Пример работы токенизатора

Предложение 1: <bos> Женщины с воплем бежали к оставленным детям; мужчины, вооружась чем попало...
   Токены: ['<bos>', 'Же', 'н', 'щи', 'ны', 'с', 'во', 'пле', 'м', 'бе', 'жали', 'к', 'оста', 'вле', 'нным', 'дет', 'я', 'м', ';', 'муж']...
   IDs:    [2, 2415, 175, 344, 252, 179, 233, 529, 174, 274]...
   Количество токенов: 128

Предложение 2: <bos> - Медведь издох - нет сомнения, но он изломал своего противника, и в этом ...
   Токены: ['<bos>', '-', 'Ме', 'две', 'дь', 'из', 'до', 'х', '-', 'нет', 'сомне', 'ния', ',', 'но', 'он', 'из', 'ло', 'мал', 'своего', 'против']...
   IDs:    [2, 20, 1144, 742, 372, 293, 245, 183, 20, 525]...
   Количество токенов: 150

Предложение 3: <bos> 'Дукмор, наш силач, красавец, молодец, о каких у нас до сего времени и не ...
   Токены: ['<bos>', "'", 'Ду', 'к', 'мор', ',', 'наш', 'сила', 'ч', ',', 'кра', 'са', 'ве', 'ц', ',', 'моло', 'де', 'ц', ',', 'о']...
   IDs:    [2, 14, 1220, 172, 1417, 19, 2055, 2021, 185, 19]...
   

Скан текстов: 100%|██████████| 82064/82064 [00:15<00:00, 5405.37it/s]

Кол-во <unk> токенов в текстах: 0
Доля <unk> токенов в текстах:   0.0000


4. Токенизируйте данные и подготовьте их к претрейну с длиной контекста 512 токенов в виде экземпляра класса `transformers.Dataset`.

In [66]:
from datasets import Dataset
from transformers import PreTrainedTokenizerFast


def tokenize(tokenizer):
    def _tokenize(samples):
        return tokenizer(
            samples["chunks"],
            truncation=True,
            padding="max_length",
            max_length=512,
            return_tensors="pt"
        )
    return _tokenize


dataset = Dataset.from_dict({"chunks": chunks}).map(
    tokenize(tokenizer_fast),
    batched=True,
    remove_columns=["chunks"]
)
dataset

Map: 100%|██████████| 82064/82064 [00:32<00:00, 2500.82 examples/s]


Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 82064
})

5. Инициализируйте модель ~150M параметров c произвольной decoder-only архитектурой трансформера. 

   Например, можно рассмотреть LlamaConfig с параметрами `hidden_size=1024, intermediate_size=1536, num_hidden_layers=16, num_attention_heads=16, num_key_value_heads=8`.

In [82]:
import torch
from dataclasses import dataclass
from transformers import LlamaConfig, LlamaForCausalLM
from torchinfo import summary


def init_model(tokenizer, model_config, device=None) -> LlamaForCausalLM:
    llama_config = LlamaConfig(
        vocab_size          = len(tokenizer.get_vocab()),
        hidden_size         = model_config.hidden_size,     
        intermediate_size   = model_config.intermediate_size,
        num_hidden_layers   = model_config.num_hidden_layers,
        num_attention_heads = model_config.num_attention_heads,
        num_key_value_heads = model_config.num_key_value_heads,
        pad_token_id        = tokenizer.pad_token_id,
        bos_token_id        = tokenizer.bos_token_id,
        eos_token_id        = tokenizer.eos_token_id,
        use_cache           = True,
    )
    model = LlamaForCausalLM(llama_config)
    
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)

    print(f"Модель инициализирована на устройстве: {device}")
    if device == "cuda":
        print(f"Устройство: {torch.cuda.get_device_name(0)}")

    return model


@dataclass
class ModelConfig:
    hidden_size: int = 1024
    intermediate_size: int = 1536
    num_hidden_layers: int = 16
    num_attention_heads: int = 16
    num_key_value_heads: int = 8

model = init_model(tokenizer_fast, ModelConfig)

summary(
    model,
    input_size=(1, 512),
    dtypes=['torch.LongTensor'],
    col_names=[
        "input_size",
        "output_size",
        "num_params",
    ],
    depth=3,
    device=next(model.parameters()).device,
)

Модель инициализирована на устройстве: cpu


Layer (type:depth-idx)                             Input Shape               Output Shape              Param #
LlamaForCausalLM                                   [1, 512]                  --                        --
├─LlamaModel: 1-1                                  --                        --                        --
│    └─Embedding: 2-1                              [1, 512]                  [1, 512, 1024]            3,072,000
│    └─LlamaRotaryEmbedding: 2-2                   [1, 512, 1024]            [1, 512, 64]              --
│    └─ModuleList: 2-3                             --                        --                        --
│    │    └─LlamaDecoderLayer: 3-1                 [1, 512, 1024]            [1, 512, 1024]            7,866,368
│    │    └─LlamaDecoderLayer: 3-2                 [1, 512, 1024]            [1, 512, 1024]            7,866,368
│    │    └─LlamaDecoderLayer: 3-3                 [1, 512, 1024]            [1, 512, 1024]            7,866,368
│    │    └─L

6. Чтобы оценить качество, используйте промпты:
```
test_prompts = [
    "Все мысли, которые имеют огромные последствия",
    "Сила войска зависит от его духа",
    "Мысль о том, что он принес страдания",
    "Человек сознает себя свободным",
    "Что бы ни случилось, я всегда буду",
    "Любовь мешает смерти",
    "Нет, жизнь не кончена",
    "Всякая мысль, даже самая простая",
    "Война не любезность, а самое гадкое дело",
    "Чтобы жить честно"
] 
```

Подготовьте коллбэки для валидации качества на промптах. Реализуйте обучение с помощью `Trainer`. Обратите внимание на параметры регуляризации `weight_decay`. Используйте подходящий `batch_size` — в диапазоне 64—128.

In [ ]:
from dataclasses import dataclass
import torch.nn as nn
from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForLanguageModeling, TrainerCallback
import math


TEST_PROMPTS = [
    "Все мысли, которые имеют огромные последствия",
    "Сила войска зависит от его духа",
    "Мысль о том, что он принес страдания",
    "Человек сознает себя свободным",
    "Что бы ни случилось, я всегда буду",
    "Любовь мешает смерти",
    "Нет, жизнь не кончена",
    "Всякая мысль, даже самая простая",
    "Война не любезность, а самое гадкое дело",
    "Чтобы жить честно"
]


class PromptEvalCallback(TrainerCallback):
    """
    Коллбэк для оценки качества генерации на тестовых промптах.
    Вызывается в конце каждой эпохи.
    """
    def __init__(self, tokenizer, model, prompts=TEST_PROMPTS, max_length=50, device=None, sample_params=None):
        self.tokenizer = tokenizer
        self.model = model
        self.prompts = prompts
        self.max_length = max_length
        self.sample_params = sample_params
        
    def on_epoch_end(self, args, state, control, **kwargs):
        """
        Генерирует тексты для каждого промпта и выводит их.
        """
        device = next(model.parameters()).device
        temperature = self.sample_params.temperature if self.sample_params else 0.8
        top_p = self.sample_params.top_p if self.sample_params else 0.9
        do_sample = self.sample_params.do_sample if self.sample_params else True

        self.model.eval()
        with torch.no_grad():
            for prompt in self.prompts:
                input_ids = self.tokenizer.encode(prompt, return_tensors="pt").to(device)
                output_ids = self.model.generate(
                    input_ids,
                    max_length=self.max_length,
                    temperature=temperature,
                    top_p=top_p,
                    do_sample=do_sample,
                    pad_token_id=self.tokenizer.pad_token_id,
                    eos_token_id=self.tokenizer.eos_token_id,
                )
                generated = self.tokenizer.decode(output_ids[0], skip_special_tokens=True)
                print(f"Промпт:         {prompt}")
                print(f"Сгенерировано:  {generated}\n")
        self.model.train()


class PerplexityCallback(TrainerCallback):
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics and "eval_loss" in metrics:
            metrics["perplexity"] = math.exp(metrics["eval_loss"])
            print(f"Perplexity: {metrics['perplexity']:.2f}")


def run_sft(model, tokenizer, train_dataset, eval_dataset, sample_params):
    cfg = SFTConfig(
        output_dir="pretarined_sft",
        per_device_train_batch_size=1,
        logging_steps=1,
        max_length=512,
        report_to='none',
        run_name='SFT',
        num_train_epochs=1,
        lr_scheduler_type="cosine",
        eval_strategy="epoch",
        learning_rate=1e-4,
        weight_decay=1e-2,
        seed=42,
    )
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False,
    )
    prompt_callback = PromptEvalCallback(
        tokenizer,
        model,
        max_length=512,
        sample_params=sample_params,
    )
    trainer = SFTTrainer(
        model=model,
        args=cfg,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=tokenizer,
        data_collator=data_collator,
        callbacks=[prompt_callback, PerplexityCallback()],
    )
    trainer.train()

In [103]:
small_dataset = dataset.select(range(10))
split = small_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]


@dataclass
class SampleParams:
    temperature: float = 0.8
    top_p: float = 0.9
    do_sample: bool = True


run_sft(model, tokenizer_fast, train_dataset, eval_dataset, SampleParams)

Truncating eval dataset: 100%|██████████| 1/1 [00:00<00:00, 551.59 examples/s]
/Users/papa/Documents/Практикум/Deep learning Engineer/dle_practicum/Sprint 6/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy,
1,7.572524,6.744152,3.689099,4608.000000,0.050000,849.078456


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.37it/s]


Промпт:         Все мысли, которые имеют огромные последствия
Сгенерировано:  Все мысли , которые име ют огром ные послед ствия , , , , , шу , , , шу , , .

Промпт:         Сила войска зависит от его духа
Сгенерировано:  Си ла вой ска зави си т от его ду ха , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , ми , , , , , , , , , шу , , , , , , , , , , , , , , , , , , , . Ро , , , шу , , шу вари , я ми , , , , , , , , Не , , , Не я , , , ми , , , , ули , , , , .

Промпт:         Мысль о том, что он принес страдания
Сгенерировано:  Мы с ль о том , что он при нес стра дания Варвара люб ъя g ступле сил , мане отправи тел g отправи , мане головой кал головой вал отправи ми шу нечего ми , шу , ца

Промпт:         Человек сознает себя свободным
Сгенерировано:  Че лове к со знает себя свобо д ным моих остро ночи уз обратился ' прият Пер Степан прият посмотрел остро прият пел остро узнать заметил остро шко посмотрел прият жно зат

/Users/papa/Documents/Практикум/Deep learning Engineer/dle_practicum/Sprint 6/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Perplexity: 849.08


## Post-train SFT
Для SFT-этапа можно использовать значительно меньше данных, поэтому возьмём модель крупнее. Рассмотрим базовую модель Qwen2.5-0.5B, с которой вы встречались в уроках. Обучите её генерировать ответы на инструктивные русскоязычные вопросы.

Для оценки качества используйте такой набор вопросов:
```
questions_rus = [
    "сколько планет в нашей солнечной системе?",
    "расскажи стих",
    "когда собирать крыжовник?",
    "Как быстро выучить новый язык?"
  ] 
```

Чтобы повысить качество ответов, проведите SFT обучение на русскоязычном инструктивном датасете [d0rj/alpaca-cleaned-ru](https://huggingface.co/datasets/d0rj/alpaca-cleaned-ru) в диалоговом формате. 

In [104]:
from datasets import load_dataset

dataset = load_dataset("d0rj/alpaca-cleaned-ru")
dataset

Generating train split: 100%|██████████| 51760/51760 [00:00<00:00, 402060.10 examples/s]


DatasetDict({
    train: Dataset({
        features: ['input', 'instruction', 'output'],
        num_rows: 51760
    })
})

In [106]:
small_dataset = dataset["train"].select(range(10))
split = small_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

In [108]:
from unsloth import FastLanguageModel
import torch

model_name = 'Qwen/Qwen2.5-0.5B'
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=512,
    load_in_8bit=True,
    load_in_4bit=False,
)

def row2messages(row):
    instruction = row['instruction']
    input = row['input']
    output = row['output']
    return [
        {"role": "system", "content": instruction},
        {"role": "user", "content": input},
        {"role": "assistant", "content": output},
    ]

ds = small_dataset.map(lambda x: {'messages': tokenizer.apply_chat_template(row2messages(x), tokenize=False)})
ds

/var/folders/_4/p98h_g953379cg5tzm5_l5p40000gn/T/ipykernel_26410/1152280479.py:1: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


NotImplementedError: Unsloth currently only works on NVIDIA, AMD and Intel GPUs.

In [109]:
from dataclasses import dataclass
import torch.nn as nn
from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForLanguageModeling, TrainerCallback
import math


questions_rus = [
    "сколько планет в нашей солнечной системе?",
    "расскажи стих",
    "когда собирать крыжовник?",
    "Как быстро выучить новый язык?"
  ] 


class PromptEvalCallback(TrainerCallback):
    """
    Коллбэк для оценки качества генерации на тестовых промптах.
    Вызывается в конце каждой эпохи.
    """
    def __init__(self, tokenizer, model, prompts=questions_rus, max_length=50, device=None, sample_params=None):
        self.tokenizer = tokenizer
        self.model = model
        self.prompts = prompts
        self.max_length = max_length
        self.sample_params = sample_params
        
    def on_epoch_end(self, args, state, control, **kwargs):
        """
        Генерирует тексты для каждого промпта и выводит их.
        """
        device = next(model.parameters()).device
        temperature = self.sample_params.temperature if self.sample_params else 0.8
        top_p = self.sample_params.top_p if self.sample_params else 0.9
        do_sample = self.sample_params.do_sample if self.sample_params else True

        self.model.eval()
        with torch.no_grad():
            for prompt in self.prompts:
                input_ids = self.tokenizer.encode(prompt, return_tensors="pt").to(device)
                output_ids = self.model.generate(
                    input_ids,
                    max_length=self.max_length,
                    temperature=temperature,
                    top_p=top_p,
                    do_sample=do_sample,
                    pad_token_id=self.tokenizer.pad_token_id,
                    eos_token_id=self.tokenizer.eos_token_id,
                )
                generated = self.tokenizer.decode(output_ids[0], skip_special_tokens=True)
                print(f"Промпт:         {prompt}")
                print(f"Сгенерировано:  {generated}\n")
        self.model.train()


class PerplexityCallback(TrainerCallback):
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics and "eval_loss" in metrics:
            metrics["perplexity"] = math.exp(metrics["eval_loss"])
            print(f"Perplexity: {metrics['perplexity']:.2f}")


def run_sft(model, tokenizer, train_dataset, eval_dataset, sample_params):
    cfg = SFTConfig(
        output_dir="posttrained_sft",
        per_device_train_batch_size=1,
        logging_steps=1,
        max_length=512,
        report_to='none',
        run_name='SFT',
        num_train_epochs=1,
        lr_scheduler_type="cosine",
        eval_strategy="epoch",
        learning_rate=1e-4,
        weight_decay=1e-2,
        seed=42,
    )
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False,
    )
    prompt_callback = PromptEvalCallback(
        tokenizer,
        model,
        max_length=512,
        sample_params=sample_params,
    )
    trainer = SFTTrainer(
        model=model,
        args=cfg,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=tokenizer,
        data_collator=data_collator,
        callbacks=[prompt_callback, PerplexityCallback()],
    )
    trainer.train()

In [ ]:
small_dataset = dataset.select(range(10))
split = small_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]


@dataclass
class SampleParams:
    temperature: float = 0.8
    top_p: float = 0.9
    do_sample: bool = True


run_sft(model, tokenizer_fast, train_dataset, eval_dataset, SampleParams)